# ROGII - Kaggle Inference (Internet OFF)

Offline inference runner for Kaggle Submit. Uses mounted `rogii-repo-v2` and `rogii-models-v2` datasets plus the competition input. No git clone, no pip install, no training.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')
REPO_ROOT = Path('/kaggle/working/repo')
REPO_DATASET_SLUG = 'rogii-repo-v2'
MODEL_DATASET_SLUG = 'rogii-models-v2'
MODEL_FILE = 'baseline_lgbm.pkl'
COMPETITION_SLUG = 'rogii-wellbore-geology-prediction'

def _path_score(path, preferred_slug):
    text = str(path).replace('\\', '/').lower()
    return (0 if preferred_slug.lower() in text else 1, len(path.parts), text)

def find_in_dataset(input_root, preferred_slug, *marker_files):
    """Find a dataset root containing the given marker files."""
    candidates = []
    for current, dirs, files in os.walk(input_root):
        dirs[:] = [d for d in dirs if d not in {'.git', '__pycache__', '.ipynb_checkpoints', '__MACOSX'}]
        path = Path(current)
        if all((path / f).exists() for f in marker_files):
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError(f'Could not find dataset [{preferred_slug}] under {input_root}')
    return sorted(candidates, key=lambda p: _path_score(p, preferred_slug))[0]

def find_zip(input_root):
    candidates = [p for p in input_root.rglob('repo.zip') if p.is_file()]
    if not candidates:
        raise FileNotFoundError(f'Could not find repo.zip; attach {REPO_DATASET_SLUG}')
    return sorted(candidates, key=lambda p: _path_score(p, REPO_DATASET_SLUG))[0]

def find_model_path(input_root):
    candidates = [p for p in input_root.rglob(MODEL_FILE) if p.is_file()]
    if not candidates:
        raise FileNotFoundError(f'Could not find {MODEL_FILE}; attach {MODEL_DATASET_SLUG}')
    return sorted(candidates, key=lambda p: _path_score(p, MODEL_DATASET_SLUG))[0]

def find_data_dir(input_root):
    return find_in_dataset(input_root, COMPETITION_SLUG, 'sample_submission.csv', 'test')

# --- Extract repo.zip ---
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
REPO_ROOT.mkdir(parents=True)

zip_path = find_zip(INPUT_ROOT)
print('Extracting repo:', zip_path)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(REPO_ROOT)
print('Extracted to:', REPO_ROOT)

# Find actual repo root (may be nested inside extracted dir)
if (REPO_ROOT / 'scripts' / 'run_predict.py').is_file():
    repo_root = REPO_ROOT
else:
    repo_root = find_in_dataset(REPO_ROOT, REPO_DATASET_SLUG, 'scripts/run_predict.py', 'src/rogii')

sys.path.insert(0, str(repo_root / 'src'))

# --- Find model and data ---
model_path = find_model_path(INPUT_ROOT)
data_dir = find_data_dir(INPUT_ROOT)

print('Repo root:', repo_root)
print('Model path:', model_path)
print('Data dir:', data_dir)
print('Output path:', OUTPUT_PATH)

# --- Run prediction with Savgol smoothing ---
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'run_predict.py'),
    '--data-dir', str(data_dir),
    '--model', str(model_path),
    '--output', str(OUTPUT_PATH),
    '--savgol-smooth',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=repo_root, check=True)

if not OUTPUT_PATH.is_file():
    raise FileNotFoundError(f'Missing expected submission file: {OUTPUT_PATH}')
if OUTPUT_PATH.stat().st_size <= 0:
    raise ValueError(f'Empty submission file: {OUTPUT_PATH}')
print(f'Submission ready: {OUTPUT_PATH} ({OUTPUT_PATH.stat().st_size} bytes)')
